<a href="https://colab.research.google.com/github/priyansh963/depression-detection-lt-edi-2022/blob/main/LT_EDI_DepRoBERTa_exact_replication_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LT-EDI 2022 DepRoBERTa baseline: exact replication

Runs the **original authors' code unmodified** (`models/finetune.py` from `rafalposwiata/depression-detection-lt-edi-2022`) with the **library versions pinned in their README**:

| package | version |
|---|---|
| Python | 3.8 |
| transformers | 4.13.0 |
| simpletransformers | 0.63.7 |
| pandas | 1.2.5 |
| scikit-learn | 0.23.1 |
| tqdm | 4.62.3 |

Their settings (from `models/config.py`) are used as they are: lr 5e-6, batch 16, 10 epochs, max length 300, weight decay 0.1, dropout 0.1, dev evaluation every 100 steps. Model: `rafalposwiata/deproberta-large-v1`. Data: `data/preprocessed_dataset`.

**Before running:** Runtime > Change runtime type > **GPU** (T4). The old stack runs in its own Python 3.8 environment, so it does not touch Colab's Python.

**Deviations (all disclosed):** `RUNS`, and a memory workaround for the T4 (batch 8 with gradient accumulation 2, which keeps the effective batch at the authors' 16 and the same number of optimizer steps). `RUNS` is the one you control. The authors run 5 repeats (v1 to v5) and report the best. One run is roughly 1.5 to 3 hours on a T4, so the default is 1. Set `RUNS = 5` to follow their full protocol (it will not fit in one free Colab session).

In [1]:
RUNS = 1   # the original repo uses 5. v1 is identical to their first run either way.

## 1. GPU check and isolated Python 3.8 environment

In [2]:
!nvidia-smi -L
!pip install -q uv
!uv venv --python 3.8 /content/venv38
!uv pip install --python /content/venv38/bin/python "numpy<1.24" "pandas==1.2.5" "scikit-learn==0.23.1" "tqdm==4.62.3" "transformers==4.13.0" "simpletransformers==0.63.7" "huggingface_hub==0.11.1" "torch==1.13.1" "protobuf<4"
!/content/venv38/bin/python -c "import torch, transformers, simpletransformers, pandas, sklearn; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__, '| pandas', pandas.__version__, '| sklearn', sklearn.__version__)"

GPU 0: Tesla T4 (UUID: GPU-831bce7f-c685-721e-65bb-0eb7d12d17fe)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 57.0 MB/s eta 0:00:00
Using CPython 3.8.20
Creating virtual environment at: venv38
Activate with: source venv38/bin/activate
Using Python 3.8.20 environment at: venv38
Resolved 107 packages in 2.99s
Prepared 105 packages in 39.28s
Installed 107 packages in 420ms
 + absl-py==2.3.1
 + aiohappyeyeballs==2.4.4
 + aiohttp==3.10.11
 + aiosignal==1.3.1
 + altair==5.4.1
 + annotated-types==0.7.0
 + async-timeout==5.0.1
 + attrs==25.3.0
 + backports-zoneinfo==0.2.1
 + blinker==1.8.2
 + cachetools==5.5.2
 + certifi==2026.7.22
 + cffi==1.17.1
 + charset-normalizer==3.5.2
 + click==8.1.8
 + cryptography==47.0.0
 + datasets==2.13.2
 + dill==0.3.6
 + eval-type-backport==0.4.0
 + filelock==3.16.1
 + frozenlist==1.5.0
 + fsspec==2025.3.0
 + gitdb==4.0.12
 + gitpython==3.2.0
 + google-auth==2.50.0
 + google-auth-oauthlib==1.0.0
 + grpcio==1.70.0
 + huggingface-hub==0.11.1
 + idna==3

`huggingface_hub==0.11.1` and `protobuf<4` are not in the authors' README. They are needed only so the 2021-era `transformers` can still import on today's package index. They do not affect training.

## 2. Get the authors' code and data

In [3]:
import os
if not os.path.exists('/content/repo'):
    !git clone -q https://github.com/rafalposwiata/depression-detection-lt-edi-2022.git /content/repo
!cd /content/repo && git log -1 --format='commit %h | %ad'
!cp /content/repo/models/config.py /content/config_original.py
!grep -n "runs" /content/repo/models/config.py

commit f3d923b | Fri May 12 15:26:32 2023 +0200
7:        self.runs: int = 5


In [4]:
# Two small edits to config.py, nothing else in the repo is touched:
# 1) number of repeats (GlobalConfig.runs)
# 2) T4 memory: batch 16 does not fit in 15 GB, so use batch 8 x gradient accumulation 2 (effective batch stays 16)
p = '/content/repo/models/config.py'
t = open(p).read()
t = t.replace('self.runs: int = 5', f'self.runs: int = {RUNS}')
t = t.replace('    model_args.train_batch_size = 16' + chr(10), '    model_args.train_batch_size = 8' + chr(10) + '    model_args.gradient_accumulation_steps = 2' + chr(10))
open(p, 'w').write(t)
!diff /content/config_original.py /content/repo/models/config.py ; echo '(diff above is the only change to the authors code)'
!grep -n "learning_rate\|train_batch_size\|num_train_epochs\|max_seq_length\|weight_decay\|evaluate_during_training_steps" /content/repo/models/config.py | head -8

7c7
<         self.runs: int = 5
---
>         self.runs: int = 1
20c20,21
<     model_args.train_batch_size = 16
---
>     model_args.train_batch_size = 8
>     model_args.gradient_accumulation_steps = 2
(diff above is the only change to the authors code)
19:    model_args.learning_rate = 5e-6
20:    model_args.train_batch_size = 8
22:    model_args.num_train_epochs = 10
23:    model_args.evaluate_during_training_steps = 100
24:    model_args.max_seq_length = 300
25:    model_args.weight_decay = 0.1
36:    lm_args.learning_rate = 4e-5
37:    lm_args.train_batch_size = 50


### 2b. Download the model locally (needed because the old transformers cannot follow Hugging Face's current redirects)

The authors' code asks for `rafalposwiata/deproberta-large-v1`. Their 2021-era `transformers` fails to download it (`MissingSchema ... No scheme supplied`). We download the **same model files** with a modern `huggingface_hub` into the folder `models/rafalposwiata/deproberta-large-v1`, which `from_pretrained` uses as a local copy. No repo file is edited.

In [5]:
import os
from huggingface_hub import snapshot_download
local = '/content/repo/models/rafalposwiata/deproberta-large-v1'
snapshot_download('rafalposwiata/deproberta-large-v1', local_dir=local,
                  ignore_patterns=['*.safetensors', '*.h5', '*.msgpack', '*.onnx', '*.ot'])
print(sorted(os.listdir(local)))
assert os.path.exists(f'{local}/pytorch_model.bin'), 'pytorch_model.bin missing'

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

['.cache', '.gitattributes', 'README.md', 'config.json', 'merges.txt', 'pytorch_model.bin', 'special_tokens_map.json', 'tokenizer_config.json', 'vocab.json']


The model's `tokenizer_config.json` contains a `tokenizer_file` entry that points to a path on the original author's machine, which does not exist here. We drop that one entry so the tokenizer is built from `vocab.json` and `merges.txt` (same tokenizer, no repo code touched), then check that it loads.

In [6]:
import json
p = f'{local}/tokenizer_config.json'
cfg = json.load(open(p)); print('tokenizer_file was:', cfg.get('tokenizer_file'))
cfg.pop('tokenizer_file', None)
json.dump(cfg, open(p, 'w'), indent=2)

tokenizer_file was: /home/depression/lm/reddit_large_roberta-large_v4/cache/tokenizer/e16a2590deb9e6d73711d6e05bf27d832fa8c1162d807222e043ca650a556964.fc9576039592f026ad76a1c231b89aee8668488c671dfbe6616bab2ed298d730


In [7]:
!/content/venv38/bin/python -c "from transformers import AutoTokenizer; t=AutoTokenizer.from_pretrained('/content/repo/models/rafalposwiata/deproberta-large-v1'); print(type(t).__name__, len(t), t('hello world')['input_ids'])"

RobertaTokenizerFast 50265 [0, 42891, 232, 2]


## 3. Fine-tune with the authors' own script

Equivalent to `cd models && python finetune.py --models DepRoBERTa`. It trains `RUNS` times (v1, v2, ...) and picks the best checkpoint by dev macro-F1 using their `find_best_checkpoint`.

Keep this tab open. Checkpoints are written to the Colab disk (about 40 GB at peak before their `clean()` step), so a disconnect means starting again.

In [8]:
!cd /content/repo/models && PYTHONPATH=..:. PYTHONUNBUFFERED=1 /content/venv38/bin/python finetune.py --models DepRoBERTa

Fine-tuning	roberta	rafalposwiata/deproberta-large-v1	v1
Some weights of the model checkpoint at rafalposwiata/deproberta-large-v1 were not used when initializing RobertaForSequenceClassification: ['lm_head.layer_norm.bias', 'lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at rafalposwiata/deproberta-large-v1 and are newly initialized: ['classifier.dense.bias',

In [12]:
import pandas as pd
d = pd.read_csv('/content/repo/models/trained_modelsrafalposwiata/deproberta-large-v1_v1/training_progress_scores.csv')
print(d[['global_step', 'f1']].tail(10))
print('best so far:', d.loc[d.f1.idxmax(), ['global_step', 'f1']].to_dict())

   global_step        f1
0          100  0.356159
1          200  0.387294
2          300  0.469320
3          375  0.552474
4          400  0.505084
5          500  0.551013
6          600  0.557840
7          700  0.565252
best so far: {'global_step': 700.0, 'f1': 0.5652524197430057}


## 4. Results

This evaluates exactly like the authors' `eval.py` (their `find_best_checkpoint`, macro-averaged metrics, `eval_batch_size=50`), but on **both dev and test**. `eval.py` itself is hard-coded to dev only.

If their `find_best_checkpoint` finds no saved checkpoint for the best-F1 step, the script falls back to the `best_model` folder that simpletransformers writes during training, and prints that it did so.

In [9]:
%%writefile /content/repo/models/eval_dev_test.py
import os, sys, json
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from simpletransformers.classification import ClassificationModel
from config import get_fine_tuning_args
from models_list import get_models
from utils import find_best_checkpoint
from dataset.utils import get_preprocessed_data, labels

version = sys.argv[1]
model_info = get_models('DepRoBERTa')[0]
model_info.model_version = version
model_args, _ = get_fine_tuning_args(model_info)
model_args.eval_batch_size = 50
out = model_args.output_dir

prog = pd.read_csv(os.path.join(out, 'training_progress_scores.csv'))
best = prog.loc[prog['f1'].idxmax()]
print(f"[{version}] best dev macro-F1 during training: {best['f1']:.4f} at step {int(best['global_step'])}")

ckpt = find_best_checkpoint(out)
if ckpt is not None:
    path, source = os.path.join(out, ckpt), f'checkpoint {ckpt} (authors find_best_checkpoint)'
else:
    path, source = os.path.join(out, 'best_model'), 'best_model folder (fallback: no checkpoint saved for the best-F1 step)'
print(f'[{version}] evaluating: {source}')

model = ClassificationModel(model_info.model_type, path, num_labels=len(labels), args=model_args)
res = {'version': version, 'best_dev_f1_during_training': round(float(best['f1']), 3),
       'best_step': int(best['global_step']), 'evaluated_model': source}
for split in ['dev', 'test']:
    data = get_preprocessed_data(split)
    preds, _ = model.predict(list(data['text'].values))
    y = data['labels'].values
    res[split] = {'acc': round(accuracy_score(y, preds), 3),
                  'precision': round(precision_score(y, preds, average='macro'), 3),
                  'recall': round(recall_score(y, preds, average='macro'), 3),
                  'f1': round(f1_score(y, preds, average='macro'), 3)}
    if split == 'test':
        print(classification_report(y, preds, target_names=labels, digits=3))
        print(confusion_matrix(y, preds))
print(json.dumps(res, indent=2))
json.dump(res, open(f'/content/results_{version}.json', 'w'), indent=2)

Writing /content/repo/models/eval_dev_test.py


In [10]:
for v in range(1, RUNS + 1):
    !cd /content/repo/models && PYTHONPATH=..:. PYTHONUNBUFFERED=1 /content/venv38/bin/python eval_dev_test.py v{v}

[v1] best dev macro-F1 during training: 0.5653 at step 700
[v1] evaluating: checkpoint checkpoint-700 (authors find_best_checkpoint)
  0% 2/1000 [00:01<10:26,  1.59it/s]
100% 20/20 [00:16<00:00,  1.20it/s]
  0% 7/3245 [00:03<23:56,  2.25it/s]
100% 65/65 [00:53<00:00,  1.23it/s]
                precision    recall  f1-score   support

        severe      0.503     0.338     0.404       228
      moderate      0.738     0.828     0.780      2169
not depression      0.514     0.399     0.449       848

      accuracy                          0.681      3245
     macro avg      0.585     0.521     0.544      3245
  weighted avg      0.663     0.681     0.667      3245

[[  77  143    8]
 [  61 1796  312]
 [  15  495  338]]
{
  "version": "v1",
  "best_dev_f1_during_training": 0.565,
  "best_step": 700,
  "evaluated_model": "checkpoint checkpoint-700 (authors find_best_checkpoint)",
  "dev": {
    "acc": 0.65,
    "precision": 0.652,
    "recall": 0.552,
    "f1": 0.565
  },
  "test": {
   

In [11]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob('/content/results_v*.json')):
    r = json.load(open(f))
    rows.append({'run': r['version'], 'best dev F1 (training log)': r['best_dev_f1_during_training'],
                 'dev F1': r['dev']['f1'], 'test acc': r['test']['acc'],
                 'test precision': r['test']['precision'], 'test recall': r['test']['recall'], 'test macro-F1': r['test']['f1']})
summary = pd.DataFrame(rows); display(summary)
print('Paper reports dev macro-F1 0.583 for DepRoBERTa. Our earlier reproduction reached 0.619 (dev).')

,run,best dev F1 (training log),dev F1,test acc,test precision,test recall,test macro-F1
0,v1,0.565,0.565,0.681,0.585,0.521,0.544


Paper reports dev macro-F1 0.583 for DepRoBERTa. Our earlier reproduction reached 0.619 (dev).


## 5. Notes for the write-up

- Dev F1 is the number the original paper tables (best of 5 runs). Test F1 is a separate measurement on a differently balanced split (see the EDA), so a drop is expected and is not an error.
- Report macro-F1, not accuracy, because of the class imbalance.
- If you ran fewer than 5 repeats, say so next to the number.